# Lesson 5.5: Follow filtering, vectorisation and ranking

Read the matching lesson at `modules/05-rag/05-source-walkthrough.md`. Run this notebook from top to bottom in the project `.venv` kernel. All cases are fictional. The default path sends no model-provider requests. A kernel retains values between cells; restart and run all after an experiment to check independence.

## The practical question

A policy assistant needs the current permitted evidence, not simply a plausible paragraph. At Harbour an older policy, another region's policy and a current policy can share many words, making relevance alone an inadequate selection rule.

In [ ]:
from ai_orgdesign.labs import m05 as lab
print("Module 5 code loaded; no provider called.")

## Follow the mechanism

retrieve first selects records with the requested tenant and active status. Only those records enter TfidfVectorizer.fit_transform. TF-IDF gives words weights based on their frequency within and across the selected documents; it creates sparse numerical vectors, not neural embeddings. The query uses the fitted vocabulary through transform. Cosine similarity compares vector direction, and ranking uses score followed by document ID to break ties reproducibly. Positive-score results retain document identity and version. grounded_context formats that evidence for a later answer stage, but performs no generation.

In [ ]:
import inspect
print(inspect.getsource(lab.retrieve))

In [ ]:
import inspect
print(inspect.getsource(lab.grounded_context))

## Predict and run

Searching delivery evidence in north returns the active P-2 policy. It excludes the retired P-1 and the south P-3 before fitting or ranking. A query whose words do not occur in the permitted vocabulary returns no hits. Returning no evidence is different from answering that no policy exists.

In [ ]:
from ai_orgdesign.labs.m05 import retrieve, grounded_context
hits = retrieve("delivery evidence")
assert [hit["id"] for hit in hits] == ["P-2"]
print(grounded_context(hits))

## Exercise

Ask whether filtering after a global top-two search can lose the best permitted result. Then remove all permitted documents and decide what the service should show.

Pause and write the predicted result and its interpretation before opening the worked answer below.

## Worked answer

A global top-two may contain only forbidden or obsolete records. Removing them afterward cannot recover the permitted records that never entered the shortlist. An empty permitted corpus returns an empty list in this lab. The interface should report that evidence was not available and route the question appropriately, rather than invent an answer.

In [ ]:
from ai_orgdesign.labs.m05 import retrieve
assert retrieve("unmatchedzzzz") == []
assert retrieve("delivery", tenant="unknown") == []
print("No permitted evidence is a visible outcome.")

## Alternatives and organisational meaning

Keyword search is easy to inspect but can miss paraphrases. Dense embeddings may recover semantic matches but require their own model, index lifecycle and evaluation. Hybrid retrieval combines signals. None reconciles conflicting authority automatically. Chunking and reranking should respond to observed failures, not an assumed need for a more elaborate stack.

Explain separately what executed, what its output establishes and what remains a question for the surrounding service.